# Practical 8 â€” Aggregations and the Analytics Report

**Objective:** use Elasticsearch aggregations to summarise the product and employee datasets
without fetching every document into Python.

**Tasks**

1. Count products by category.
2. Calculate average product price.
3. Calculate maximum and minimum price.
4. Calculate total inventory.
5. Find the top 10 product categories.
6. Calculate the number of products created each month.
7. Calculate average salary by employee department.
8. Count unique employees by city.
9. Create nested aggregations.
10. Perform aggregations with filters.

**Assignment:** build an analytics report with at least 15 aggregations and explain the
difference between bucket and metric aggregations.

## Dataset and index design

`generate_data.py` writes two deterministic files (seed 42):

| File | Documents | Highlights |
|---|---:|---|
| `products.json` | 72 | 12 categories x 6 products, `price`, `stock`, `rating`, `created_at`, 12 brands, `reviews` (nested), `supplier` |
| `employees.json` | 120 | 7 departments, 8 cities, `salary`, sparse `bonus`, `active`, `joining_date` |

Both indices are created explicitly so aggregation fields are `keyword`, numeric, boolean
or date as required. `reviews` is a **nested** field, which is what makes Task 9 meaningful:
nested aggregations see one review per nested document instead of one flattened array per product.

In [1]:
import json
import math
import statistics
from collections import Counter
from pathlib import Path

from elasticsearch import Elasticsearch, helpers

es = Elasticsearch("http://127.0.0.1:9200", request_timeout=60)
info = es.info()
print("Connected to Elasticsearch", info["version"]["number"])
print("cluster:", info["cluster_name"], "| node:", info["name"])

HERE = Path.cwd()
PRODUCTS = json.loads((HERE / "products.json").read_text(encoding="utf-8"))
EMPLOYEES = json.loads((HERE / "employees.json").read_text(encoding="utf-8"))

print("products loaded:", len(PRODUCTS))
print("employees loaded:", len(EMPLOYEES))
print("products by category:", dict(sorted(Counter(p["category"] for p in PRODUCTS).items())))
print("reviews stored:", sum(len(p["reviews"]) for p in PRODUCTS))

assert len(PRODUCTS) == 72
assert len(EMPLOYEES) == 120

Connected to Elasticsearch 9.2.0
cluster: my-elasticsearch-cluster | node: my-node
products loaded: 72
employees loaded: 120
products by category: {'Automotive': 6, 'Beauty': 6, 'Books': 6, 'Electronics': 6, 'Fashion': 6, 'Garden': 6, 'Grocery': 6, 'Home & Kitchen': 6, 'Office': 6, 'Pet Supplies': 6, 'Sports': 6, 'Toys': 6}
reviews stored: 181


In [2]:
PRODUCT_MAPPING = {
    "settings": {"number_of_shards": 1, "number_of_replicas": 0},
    "mappings": {
        "properties": {
            "product_id": {"type": "keyword"},
            "name": {
                "type": "text",
                "fields": {"keyword": {"type": "keyword", "ignore_above": 256}},
            },
            "description": {"type": "text"},
            "brand": {"type": "keyword"},
            "category": {"type": "keyword"},
            "subcategory": {"type": "keyword"},
            "price": {"type": "float"},
            "stock": {"type": "integer"},
            "rating": {"type": "float"},
            "tags": {"type": "keyword"},
            "available": {"type": "boolean"},
            "created_at": {"type": "date"},
            "sku": {"type": "keyword"},
            "warranty_months": {"type": "integer"},
            "supplier": {
                "properties": {
                    "name": {"type": "keyword"},
                    "country": {"type": "keyword"},
                    "lead_time_days": {"type": "integer"},
                }
            },
            "reviews": {
                "type": "nested",
                "properties": {
                    "author": {"type": "keyword"},
                    "rating": {"type": "float"},
                    "comment": {"type": "text"},
                    "date": {"type": "date"},
                },
            },
        }
    },
}

EMPLOYEE_MAPPING = {
    "settings": {"number_of_shards": 1, "number_of_replicas": 0},
    "mappings": {
        "properties": {
            "employee_id": {"type": "integer"},
            "name": {
                "type": "text",
                "fields": {"keyword": {"type": "keyword", "ignore_above": 256}},
            },
            "department": {"type": "keyword"},
            "designation": {"type": "keyword"},
            "salary": {"type": "integer"},
            "city": {"type": "keyword"},
            "joining_date": {"type": "date"},
            "active": {"type": "boolean"},
            "email": {"type": "keyword"},
            "bonus": {"type": "float"},
        }
    },
}

for index, mapping in (("products", PRODUCT_MAPPING), ("employees", EMPLOYEE_MAPPING)):
    if es.indices.exists(index=index):
        es.indices.delete(index=index)
    es.indices.create(index=index, **mapping)

helpers.bulk(
    es,
    [{"_index": "products", "_id": p["product_id"], "_source": p} for p in PRODUCTS],
)
helpers.bulk(
    es,
    [{"_index": "employees", "_id": e["employee_id"], "_source": e} for e in EMPLOYEES],
)
es.indices.refresh(index="products,employees")

product_total = es.count(index="products")["count"]
employee_total = es.count(index="employees")["count"]
print("products index:", product_total, "documents")
print("employees index:", employee_total, "documents")
print("reviews mapping type:", es.indices.get_mapping(index="products")["products"]["mappings"]["properties"]["reviews"]["type"])

assert product_total == len(PRODUCTS) == 72
assert employee_total == len(EMPLOYEES) == 120

products index: 72 documents
employees index: 120 documents
reviews mapping type: nested


## Helper functions

Every task uses the same shape: a `size: 0` search (no hits, only aggregations), then a
small helper that prints the buckets. Assertions compare Elasticsearch's answer with the
same calculation done in Python from the source JSON, so the notebook either proves the
aggregation or stops with an error.

In [3]:
def buckets_of(aggregation):
    """Return a list of buckets for both list-style and keyed (filters) buckets."""
    buckets = aggregation["buckets"]
    if isinstance(buckets, dict):
        return [{"key": key, **value} for key, value in buckets.items()]
    return buckets


def value_of(bucket, value_key):
    raw = bucket[value_key]
    if isinstance(raw, dict):
        return raw.get("value", raw.get("doc_count"))
    return raw


def show_buckets(title, aggregation, value_label=None, value_key=None, digits=2):
    buckets = buckets_of(aggregation)
    print(title)

    header = f"{'bucket':<26}{'doc_count':>10}"
    if value_key:
        header += f"{value_label:>18}"
    print(header)
    print("-" * len(header))

    for bucket in buckets:
        key = bucket.get("key_as_string", bucket["key"])
        line = f"{str(key):<26}{bucket['doc_count']:>10}"
        if value_key:
            value = value_of(bucket, value_key)
            shown = "-" if value is None else f"{value:,.{digits}f}"
            line += f"{shown:>18}"
        print(line)

    print("-" * len(header))
    print(f"{'TOTAL / BUCKETS':<26}{sum(b['doc_count'] for b in buckets):>10}"
          f"{('  (' + str(len(buckets)) + ' buckets)'):>18}")
    print()


def show_metric(label, value, digits=4):
    if value is None:
        text = "null"
    elif isinstance(value, float):
        text = f"{value:,.{digits}f}"
    else:
        text = f"{value:,}"
    print(f"{label:<34} {text}")


print("aggregation helpers ready")

aggregation helpers ready


## Task 1 â€” Count products by category

`terms` is the classic **multi-bucket** aggregation: one bucket per distinct `category`,
ordered by `doc_count` by default. Because the field is mapped as `keyword`, the full
category name (for example `Home & Kitchen`) is one term, not two analyzed words.

In [4]:
aggs = {"by_category": {"terms": {"field": "category", "size": 20}}}
response = es.search(index="products", size=0, aggs=aggs)
by_category = response["aggregations"]["by_category"]

show_buckets("Products by category", by_category)

counts = {b["key"]: b["doc_count"] for b in by_category["buckets"]}
expected = dict(Counter(p["category"] for p in PRODUCTS))

print("buckets:", len(counts), "| categories:", len(expected))
print("sum_other_doc_count:", by_category["sum_other_doc_count"])
print("total products:", sum(counts.values()))

assert counts == expected
assert sum(counts.values()) == len(PRODUCTS) == 72
assert by_category["sum_other_doc_count"] == 0
assert len(counts) == 12

Products by category
bucket                     doc_count
------------------------------------
Automotive                         6
Beauty                             6
Books                              6
Electronics                        6
Fashion                            6
Garden                             6
Grocery                            6
Home & Kitchen                     6
Office                             6
Pet Supplies                       6
Sports                             6
Toys                               6
------------------------------------
TOTAL / BUCKETS                   72      (12 buckets)

buckets: 12 | categories: 12
sum_other_doc_count: 0
total products: 72


## Task 2 â€” Average product price

`avg` is a **single-value metric** aggregation. It walks all matching documents and returns
one number, so no documents are transferred to the client.

In [5]:
response = es.search(
    index="products",
    size=0,
    aggs={"avg_price": {"avg": {"field": "price"}}},
)
avg_price = response["aggregations"]["avg_price"]["value"]
python_avg = statistics.mean(p["price"] for p in PRODUCTS)

show_metric("Elasticsearch avg price", avg_price)
show_metric("Python avg price", python_avg)
show_metric("difference", abs(avg_price - python_avg))

assert math.isclose(avg_price, python_avg, rel_tol=1e-6)
print("\navg matches Python within float precision; sum =", round(sum(p["price"] for p in PRODUCTS), 2))

Elasticsearch avg price            5,112.5067
Python avg price                   5,112.5067
difference                         0.0000

avg matches Python within float precision; sum = 368100.48


## Task 3 â€” Maximum and minimum price

`min` and `max` are separate metric aggregations. `stats` is a convenient **multi-value
metric** that returns `count`, `min`, `max`, `avg` and `sum` in one request.

In [6]:
response = es.search(
    index="products",
    size=0,
    aggs={
        "min_price": {"min": {"field": "price"}},
        "max_price": {"max": {"field": "price"}},
        "price_stats": {"stats": {"field": "price"}},
    },
)
result = response["aggregations"]
minimum = result["min_price"]["value"]
maximum = result["max_price"]["value"]
stats = result["price_stats"]

show_metric("min_price", minimum)
show_metric("max_price", maximum)
print()
show_metric("stats.count", stats["count"])
show_metric("stats.min", stats["min"])
show_metric("stats.max", stats["max"])
show_metric("stats.avg", stats["avg"])

prices = [p["price"] for p in PRODUCTS]
assert minimum == min(prices)
assert maximum == max(prices)
assert stats["count"] == len(prices)
assert math.isclose(stats["avg"], statistics.mean(prices), rel_tol=1e-6)
print("\nmin/max/stats all match Python; the most expensive product costs", maximum)

min_price                          9.5000
max_price                          78,999.0000

stats.count                        72
stats.min                          9.5000
stats.max                          78,999.0000
stats.avg                          5,112.5067

min/max/stats all match Python; the most expensive product costs 78999.0


## Task 4 â€” Total inventory

`sum` adds the `stock` field across all products. "Total inventory" here means **units in
stock** (sum of quantities), not the number of product documents.

In [7]:
response = es.search(
    index="products",
    size=0,
    aggs={"total_inventory": {"sum": {"field": "stock"}}},
)
total_inventory = response["aggregations"]["total_inventory"]["value"]
python_inventory = sum(p["stock"] for p in PRODUCTS)

show_metric("Elasticsearch total stock", total_inventory)
show_metric("Python total stock", python_inventory)
show_metric("products in stock", sum(1 for p in PRODUCTS if p["stock"] > 0))
show_metric("products out of stock", sum(1 for p in PRODUCTS if p["stock"] == 0))

assert total_inventory == python_inventory == 5753
print("\nsum ignores documents without the field; all 72 products carry stock")

Elasticsearch total stock          5,753.0000
Python total stock                 5,753
products in stock                  66
products out of stock              6

sum ignores documents without the field; all 72 products carry stock


## Task 5 â€” Top 10 product categories

`size: 10` keeps the ten largest buckets and reports the rest in `sum_other_doc_count`.
With 72 products split evenly across 12 categories, every category has 6 products, so the
top ten are the first ten keys in alphabetical order; the two omitted categories are counted
in `sum_other_doc_count`.

In [8]:
response = es.search(
    index="products",
    size=0,
    aggs={"top_categories": {"terms": {"field": "category", "size": 10}}},
)
top_categories = response["aggregations"]["top_categories"]

show_buckets("Top 10 categories (default order: doc_count desc, key asc)", top_categories, "doc_count", None)

print("returned buckets:", len(top_categories["buckets"]))
print("documents in returned buckets:", sum(b["doc_count"] for b in top_categories["buckets"]))
print("sum_other_doc_count:", top_categories["sum_other_doc_count"])

assert len(top_categories["buckets"]) == 10
assert sum(b["doc_count"] for b in top_categories["buckets"]) == 60
assert top_categories["sum_other_doc_count"] == 12
print("\n12 categories exist; size=10 hides 2 of them and sums their 12 documents")

Top 10 categories (default order: doc_count desc, key asc)
bucket                     doc_count
------------------------------------
Automotive                         6
Beauty                             6
Books                              6
Electronics                        6
Fashion                            6
Garden                             6
Grocery                            6
Home & Kitchen                     6
Office                             6
Pet Supplies                       6
------------------------------------
TOTAL / BUCKETS                   60      (10 buckets)

returned buckets: 10
documents in returned buckets: 60
sum_other_doc_count: 12

12 categories exist; size=10 hides 2 of them and sums their 12 documents


## Task 6 â€” Products created each month

`date_histogram` is a bucket aggregation that groups dates into calendar months. Two options
make the result report-friendly:

- `min_doc_count: 0` keeps months with no products.
- `extended_bounds` sets the full reporting window (January 2023 â€“ December 2025).

The dataset deliberately leaves six months empty so the zero buckets are visible.

In [9]:
response = es.search(
    index="products",
    size=0,
    aggs={
        "monthly_created": {
            "date_histogram": {
                "field": "created_at",
                "calendar_interval": "month",
                "format": "yyyy-MM",
                "min_doc_count": 0,
                "extended_bounds": {"min": "2023-01", "max": "2025-12"},
            },
            "aggs": {"avg_price": {"avg": {"field": "price"}}},
        }
    },
)
monthly = response["aggregations"]["monthly_created"]

show_buckets("Products created per month", monthly, "avg price", "avg_price")

months = [b["key_as_string"] for b in monthly["buckets"]]
empty_months = [b["key_as_string"] for b in monthly["buckets"] if b["doc_count"] == 0]
python_monthly = Counter(p["created_at"][:7] for p in PRODUCTS)

print("first / last month:", months[0], "/", months[-1])
print("buckets:", len(months), "| documents:", sum(b["doc_count"] for b in monthly["buckets"]))
print("empty months:", empty_months)

assert len(months) == 36
assert months[0] == "2023-01" and months[-1] == "2025-12"
assert sum(b["doc_count"] for b in monthly["buckets"]) == 72
assert empty_months == ["2023-02", "2023-07", "2024-01", "2024-08", "2025-03", "2025-09"]

for bucket in monthly["buckets"]:
    month = bucket["key_as_string"]
    assert bucket["doc_count"] == python_monthly.get(month, 0)
    if bucket["doc_count"]:
        expected_avg = statistics.mean(
            p["price"] for p in PRODUCTS if p["created_at"][:7] == month
        )
        assert math.isclose(bucket["avg_price"]["value"], expected_avg, rel_tol=1e-6)

print("\nall 36 months returned; six zero buckets prove min_doc_count/extended_bounds work")

Products created per month
bucket                     doc_count         avg price
------------------------------------------------------
2023-01                            3          1,032.66
2023-02                            0                 -
2023-03                            3         28,115.67
2023-04                            3         21,332.33
2023-05                            3          3,019.33
2023-06                            3         10,932.33
2023-07                            0                 -
2023-08                            3         18,499.00
2023-09                            3          1,669.17
2023-10                            3          2,399.00
2023-11                            3          4,999.00
2023-12                            3          1,399.00
2024-01                            0                 -
2024-02                            3          1,432.33
2024-03                            3          2,532.33
2024-04                            2  

## Task 7 â€” Average salary by employee department

A metric aggregation can be **nested inside a bucket aggregation**. Here `terms` creates one
bucket per department and `avg_salary`/`salary_stats` run inside each bucket, so one request
returns the whole department report.

In [10]:
response = es.search(
    index="employees",
    size=0,
    aggs={
        "by_department": {
            "terms": {"field": "department", "size": 20},
            "aggs": {
                "avg_salary": {"avg": {"field": "salary"}},
                "salary_stats": {"stats": {"field": "salary"}},
            },
        }
    },
)
by_department = response["aggregations"]["by_department"]

show_buckets("Employees and average salary by department", by_department, "avg salary", "avg_salary", 0)

department_counts = Counter(e["department"] for e in EMPLOYEES)
for bucket in by_department["buckets"]:
    department = bucket["key"]
    salaries = [e["salary"] for e in EMPLOYEES if e["department"] == department]
    assert bucket["doc_count"] == department_counts[department]
    assert math.isclose(bucket["avg_salary"]["value"], statistics.mean(salaries), rel_tol=1e-6)
    assert bucket["salary_stats"]["min"] == min(salaries)
    assert bucket["salary_stats"]["max"] == max(salaries)

print("buckets:", len(by_department["buckets"]), "| employees:", sum(b["doc_count"] for b in by_department["buckets"]))
print("highest-paying department:",
      max(by_department["buckets"], key=lambda b: b["avg_salary"]["value"])["key"])
assert len(by_department["buckets"]) == 7
assert sum(b["doc_count"] for b in by_department["buckets"]) == 120

Employees and average salary by department
bucket                     doc_count        avg salary
------------------------------------------------------
Engineering                       18            83,139
Finance                           17            73,382
HR                                17            76,147
Marketing                         17            67,706
Operations                        17            79,147
Sales                             17            90,235
Support                           17           103,029
------------------------------------------------------
TOTAL / BUCKETS                  120       (7 buckets)

buckets: 7 | employees: 120
highest-paying department: Support


## Task 8 â€” Count unique employees by city

`cardinality` is an approximate distinct-count metric (HyperLogLog++). Because each employee
has exactly one document, the per-city cardinality of `employee_id` equals that city's
`doc_count`; the comparison below proves the approximation is exact at this scale.
`precision_threshold: 1000` raises accuracy for small/medium sets.

In [11]:
response = es.search(
    index="employees",
    size=0,
    aggs={
        "by_city": {
            "terms": {"field": "city", "size": 20},
            "aggs": {
                "unique_employees": {
                    "cardinality": {"field": "employee_id", "precision_threshold": 1000}
                }
            },
        },
        "unique_total": {
            "cardinality": {"field": "employee_id", "precision_threshold": 1000}
        },
    },
)
result = response["aggregations"]
by_city = result["by_city"]

show_buckets("Unique employees by city", by_city, "unique", "unique_employees", 0)

for bucket in by_city["buckets"]:
    assert bucket["unique_employees"]["value"] == bucket["doc_count"]

print("unique employees overall:", int(result["unique_total"]["value"]))
print("sum of city buckets:", sum(b["doc_count"] for b in by_city["buckets"]))
assert result["unique_total"]["value"] == 120
assert len(by_city["buckets"]) == 8
print("\ncardinality == doc_count because employee_id is unique per document")

Unique employees by city
bucket                     doc_count            unique
------------------------------------------------------
Pune                              22                22
Kolkata                           18                18
Mumbai                            17                17
Ahmedabad                         14                14
Bengaluru                         14                14
Delhi                             14                14
Chennai                           12                12
Hyderabad                          9                 9
------------------------------------------------------
TOTAL / BUCKETS                  120       (8 buckets)

unique employees overall: 120
sum of city buckets: 120

cardinality == doc_count because employee_id is unique per document


## Task 9 â€” Nested aggregations

`reviews` is mapped as `nested`, so Elasticsearch indexes each review as its own hidden
document. A `nested` aggregation gives metrics over those review documents. `reverse_nested`
jumps back to the parent product, which is how `products_with_author` counts **products**
rather than reviews.

If `reviews` were a plain object array, the avg/max metrics would see flattened values and
the author buckets would lose the link to the product.

In [12]:
response = es.search(
    index="products",
    size=0,
    aggs={
        "reviews": {
            "nested": {"path": "reviews"},
            "aggs": {
                "avg_review_rating": {"avg": {"field": "reviews.rating"}},
                "max_review_rating": {"max": {"field": "reviews.rating"}},
                "authors": {
                    "terms": {"field": "reviews.author", "size": 20},
                    "aggs": {
                        "products_with_author": {"reverse_nested": {}},
                    },
                },
            },
        }
    },
)
reviews = response["aggregations"]["reviews"]
all_review_ratings = [r["rating"] for p in PRODUCTS for r in p["reviews"]]

print("nested documents (reviews):", reviews["doc_count"])
print("average review rating:", round(reviews["avg_review_rating"]["value"], 4))
print("maximum review rating:", reviews["max_review_rating"]["value"])
print()

show_buckets("Review authors (nested) and parent products", reviews["authors"], "products", "products_with_author", 0)

assert reviews["doc_count"] == len(all_review_ratings) == 181
assert math.isclose(reviews["avg_review_rating"]["value"], statistics.mean(all_review_ratings), rel_tol=1e-6)
assert reviews["max_review_rating"]["value"] == max(all_review_ratings) == 5.0
assert sum(b["doc_count"] for b in reviews["authors"]["buckets"]) == 181

for bucket in reviews["authors"]["buckets"]:
    parents = bucket["products_with_author"]["doc_count"]
    assert 0 < parents <= bucket["doc_count"] <= 72

print("review buckets sum to 181; reverse_nested shows products (<= reviews) per author")

nested documents (reviews): 181
average review rating: 2.9392
maximum review rating: 5.0

Review authors (nested) and parent products
bucket                     doc_count          products
------------------------------------------------------
dave                              19                19
wendy                             14                14
mallory                           12                12
trent                             12                11
victor                            12                12
alice                             11                10
bob                               10                 9
grace                             10                 9
rupert                            10                 9
frank                              9                 8
niaj                               9                 9
carol                              8                 8
erin                               7                 7
judy                               7     

## Task 10 â€” Aggregations with filters

Three ways to restrict an aggregation:

| Form | Meaning |
|---|---|
| `filter` | one bucket containing only matching documents, then sub-aggregations |
| `filters` | several named buckets, each with its own filter |
| `global` | escape all query/filter context and aggregate the whole index |

The comparison at the end shows how the available-products average differs from the global
average in the same response.

In [13]:
response = es.search(
    index="products",
    size=0,
    aggs={
        "available_only": {
            "filter": {"term": {"available": True}},
            "aggs": {
                "avg_price": {"avg": {"field": "price"}},
                "total_stock": {"sum": {"field": "stock"}},
            },
        },
        "price_tiers": {
            "filters": {
                "filters": {
                    "budget": {"range": {"price": {"lt": 1000}}},
                    "mid": {"range": {"price": {"gte": 1000, "lt": 10000}}},
                    "premium": {"range": {"price": {"gte": 10000}}},
                }
            },
            "aggs": {"avg_rating": {"avg": {"field": "rating"}}},
        },
        "global_all": {
            "global": {},
            "aggs": {"avg_price": {"avg": {"field": "price"}}},
        },
    },
)
result = response["aggregations"]
available = result["available_only"]
tiers = result["price_tiers"]

print("available_only:")
print("  doc_count:", available["doc_count"])
print("  avg price:", round(available["avg_price"]["value"], 2))
print("  total stock:", int(available["total_stock"]["value"]))
print()

show_buckets("Price tiers (filters aggregation)", tiers, "avg rating", "avg_rating")

python_available = [p for p in PRODUCTS if p["available"]]
assert available["doc_count"] == len(python_available) == 66
assert math.isclose(available["avg_price"]["value"],
                    statistics.mean(p["price"] for p in python_available), rel_tol=1e-6)


def tier_of(price):
    if price < 1000:
        return "budget"
    if price < 10000:
        return "mid"
    return "premium"


python_tiers = Counter(tier_of(p["price"]) for p in PRODUCTS)
tier_buckets = {b["key"]: b for b in buckets_of(tiers)}
assert set(tier_buckets) == {"budget", "mid", "premium"}
for key, bucket in tier_buckets.items():
    assert bucket["doc_count"] == python_tiers[key]

global_avg = result["global_all"]["avg_price"]["value"]
overall_avg = statistics.mean(p["price"] for p in PRODUCTS)
assert math.isclose(global_avg, overall_avg, rel_tol=1e-6)

print("tier counts:", {k: v["doc_count"] for k, v in tier_buckets.items()})
print("tier sum:", sum(v["doc_count"] for v in tier_buckets.values()))
print("global avg price:", round(global_avg, 2))
print("available avg price:", round(available["avg_price"]["value"], 2))
print("difference:", round(overall_avg - available["avg_price"]["value"], 2))

assert sum(v["doc_count"] for v in tier_buckets.values()) == 72

available_only:
  doc_count: 66
  avg price: 4903.13
  total stock: 5753

Price tiers (filters aggregation)
bucket                     doc_count        avg rating
------------------------------------------------------
budget                            25              3.65
mid                               39              3.72
premium                            8              3.62
------------------------------------------------------
TOTAL / BUCKETS                   72       (3 buckets)

tier counts: {'budget': 25, 'mid': 39, 'premium': 8}
tier sum: 72
global avg price: 5112.51
available avg price: 4903.13
difference: 209.38


## Assignment â€” Analytics report with 15+ aggregations

The report below runs **two** searches (products and employees) and records every aggregation
with its index, kind (bucket/metric), aggregation type and value. It is written to
`aggregation_report.json` so the report is a reproducible artefact, not just printed output.

Aggregations included: `terms`, `avg`, `sum`, `min`, `max`, `stats`, `extended_stats`,
`value_count`, `cardinality`, `percentiles`, `date_histogram`, `histogram`, `range`,
`filters`, `filter`, `global`, `missing`, `nested` and `reverse_nested`.

In [14]:
PRODUCT_REPORT_AGGS = {
    "categories": {"terms": {"field": "category", "size": 20}},
    "top_categories": {"terms": {"field": "category", "size": 10}},
    "average_price": {"avg": {"field": "price"}},
    "minimum_price": {"min": {"field": "price"}},
    "maximum_price": {"max": {"field": "price"}},
    "price_stats": {"stats": {"field": "price"}},
    "price_percentiles": {"percentiles": {"field": "price", "percents": [25, 50, 75, 90, 95]}},
    "total_inventory": {"sum": {"field": "stock"}},
    "product_count": {"value_count": {"field": "product_id"}},
    "monthly_created": {
        "date_histogram": {
            "field": "created_at",
            "calendar_interval": "month",
            "format": "yyyy-MM",
            "min_doc_count": 0,
            "extended_bounds": {"min": "2023-01", "max": "2025-12"},
        }
    },
    "brands": {"terms": {"field": "brand", "size": 20}},
    "price_histogram": {"histogram": {"field": "price", "interval": 5000}},
    "price_ranges": {
        "range": {
            "field": "price",
            "ranges": [{"to": 1000}, {"from": 1000, "to": 10000}, {"from": 10000}],
        }
    },
    "price_tiers": {
        "filters": {
            "filters": {
                "budget": {"range": {"price": {"lt": 1000}}},
                "mid": {"range": {"price": {"gte": 1000, "lt": 10000}}},
                "premium": {"range": {"price": {"gte": 10000}}},
            }
        },
        "aggs": {"avg_rating": {"avg": {"field": "rating"}}},
    },
    "rating_extended": {"extended_stats": {"field": "rating"}},
    "available_only": {
        "filter": {"term": {"available": True}},
        "aggs": {"avg_price": {"avg": {"field": "price"}}},
    },
    "stock_zero": {"filter": {"term": {"stock": 0}}},
    "global_all": {"global": {}, "aggs": {"avg_price": {"avg": {"field": "price"}}}},
    "supplier_countries": {"terms": {"field": "supplier.country", "size": 20}},
    "price_by_category": {
        "terms": {"field": "category", "size": 20},
        "aggs": {"avg_price": {"avg": {"field": "price"}}},
    },
    "reviews": {
        "nested": {"path": "reviews"},
        "aggs": {
            "avg_review_rating": {"avg": {"field": "reviews.rating"}},
            "parent_products": {"reverse_nested": {}},
            "authors": {
                "terms": {"field": "reviews.author", "size": 20},
                "aggs": {"products_with_author": {"reverse_nested": {}}},
            },
        },
    },
}

EMPLOYEE_REPORT_AGGS = {
    "salary_by_department": {
        "terms": {"field": "department", "size": 20},
        "aggs": {"avg_salary": {"avg": {"field": "salary"}}},
    },
    "unique_by_city": {
        "terms": {"field": "city", "size": 20},
        "aggs": {
            "unique_employees": {
                "cardinality": {"field": "employee_id", "precision_threshold": 1000}
            }
        },
    },
    "unique_employee_total": {
        "cardinality": {"field": "employee_id", "precision_threshold": 1000}
    },
    "missing_bonus": {"missing": {"field": "bonus"}},
    "active_only": {
        "filter": {"term": {"active": True}},
        "aggs": {"avg_salary": {"avg": {"field": "salary"}}},
    },
    "designations": {"terms": {"field": "designation", "size": 30}},
    "salary_percentiles": {"percentiles": {"field": "salary", "percents": [25, 50, 75, 90, 95]}},
    "employee_count": {"value_count": {"field": "employee_id"}},
}

product_result = es.search(index="products", size=0, aggs=PRODUCT_REPORT_AGGS)["aggregations"]
employee_result = es.search(index="employees", size=0, aggs=EMPLOYEE_REPORT_AGGS)["aggregations"]

report_rows = []


def add_row(name, index, kind, agg_type, value):
    report_rows.append({
        "aggregation": name,
        "index": index,
        "kind": kind,
        "type": agg_type,
        "value": value,
    })


def summarize(aggregation):
    if "buckets" in aggregation:
        buckets = buckets_of(aggregation)
        top = buckets[0] if buckets else None
        return {
            "buckets": len(buckets),
            "documents": sum(b["doc_count"] for b in buckets),
            "top": None if top is None else {
                "key": top.get("key_as_string", top["key"]),
                "doc_count": top["doc_count"],
            },
        }
    return {"documents": aggregation.get("doc_count")}


add_row("categories", "products", "bucket", "terms", summarize(product_result["categories"]))
add_row("top_categories", "products", "bucket", "terms", summarize(product_result["top_categories"]))
add_row("average_price", "products", "metric", "avg", round(product_result["average_price"]["value"], 4))
add_row("minimum_price", "products", "metric", "min", round(product_result["minimum_price"]["value"], 4))
add_row("maximum_price", "products", "metric", "max", round(product_result["maximum_price"]["value"], 4))
add_row("price_stats", "products", "metric", "stats", {
    key: round(product_result["price_stats"][key], 4)
    for key in ("count", "min", "max", "avg", "sum")
})
add_row("price_percentiles", "products", "metric", "percentiles",
        {k: round(v, 4) for k, v in product_result["price_percentiles"]["values"].items()})
add_row("total_inventory", "products", "metric", "sum", product_result["total_inventory"]["value"])
add_row("product_count", "products", "metric", "value_count", product_result["product_count"]["value"])
add_row("monthly_created", "products", "bucket", "date_histogram", summarize(product_result["monthly_created"]))
add_row("brands", "products", "bucket", "terms", summarize(product_result["brands"]))
add_row("price_histogram", "products", "bucket", "histogram", summarize(product_result["price_histogram"]))
add_row("price_ranges", "products", "bucket", "range", summarize(product_result["price_ranges"]))
add_row("price_tiers", "products", "bucket", "filters", summarize(product_result["price_tiers"]))
add_row("rating_extended", "products", "metric", "extended_stats", {
    key: round(product_result["rating_extended"][key], 4)
    for key in ("count", "min", "max", "avg", "sum")
})
add_row("available_only", "products", "bucket", "filter", summarize(product_result["available_only"]))
add_row("available_avg_price", "products", "metric", "avg",
        round(product_result["available_only"]["avg_price"]["value"], 4))
add_row("stock_zero", "products", "bucket", "filter", summarize(product_result["stock_zero"]))
add_row("global_all", "products", "bucket", "global", summarize(product_result["global_all"]))
add_row("global_avg_price", "products", "metric", "avg",
        round(product_result["global_all"]["avg_price"]["value"], 4))
add_row("supplier_countries", "products", "bucket", "terms", summarize(product_result["supplier_countries"]))
add_row("price_by_category", "products", "bucket", "terms", summarize(product_result["price_by_category"]))
add_row("reviews_nested", "products", "bucket", "nested", summarize(product_result["reviews"]))
add_row("review_parent_products", "products", "bucket", "reverse_nested",
        summarize(product_result["reviews"]["parent_products"]))
add_row("avg_review_rating", "products", "metric", "avg",
        round(product_result["reviews"]["avg_review_rating"]["value"], 4))
add_row("review_authors", "products", "bucket", "terms", summarize(product_result["reviews"]["authors"]))

add_row("salary_by_department", "employees", "bucket", "terms", summarize(employee_result["salary_by_department"]))
for bucket in employee_result["salary_by_department"]["buckets"]:
    add_row(f"avg_salary_{bucket['key'].lower()}", "employees", "metric", "avg",
            round(bucket["avg_salary"]["value"], 4))
add_row("unique_by_city", "employees", "bucket", "terms", summarize(employee_result["unique_by_city"]))
add_row("unique_employee_total", "employees", "metric", "cardinality",
        int(employee_result["unique_employee_total"]["value"]))
add_row("missing_bonus", "employees", "bucket", "missing", summarize(employee_result["missing_bonus"]))
add_row("active_only", "employees", "bucket", "filter", summarize(employee_result["active_only"]))
add_row("active_avg_salary", "employees", "metric", "avg",
        round(employee_result["active_only"]["avg_salary"]["value"], 4))
add_row("designations", "employees", "bucket", "terms", summarize(employee_result["designations"]))
add_row("salary_percentiles", "employees", "metric", "percentiles",
        {k: round(v, 4) for k, v in employee_result["salary_percentiles"]["values"].items()})
add_row("employee_count", "employees", "metric", "value_count", employee_result["employee_count"]["value"])

def report_text(value):
    if isinstance(value, dict):
        if "buckets" in value:
            top = value["top"]
            text = f"{value['buckets']} buckets / {value['documents']} docs"
            if top:
                text += f" / top {top['key']} ({top['doc_count']})"
            return text
        if {"avg", "min", "max"} <= set(value):
            return f"avg {value['avg']:,.2f}, min {value['min']:,.2f}, max {value['max']:,.2f}"
        def item_text(item):
            if isinstance(item, float):
                return f"{item:,.2f}"
            if isinstance(item, int):
                return f"{item:,}"
            return str(item)

        return ", ".join(f"{key} {item_text(item)}" for key, item in value.items())
    if isinstance(value, float):
        return f"{value:,.4f}"
    return f"{value:,}"


print(f"{'#':>3} {'aggregation':<34}{'index':<10}{'kind':<8}{'type':<16}value")
print("-" * 118)
for number, row in enumerate(report_rows, 1):
    print(f"{number:>3} {row['aggregation']:<34}{row['index']:<10}"
          f"{row['kind']:<8}{row['type']:<16}{report_text(row['value'])}")

REPORT = {
    "dataset": {
        "products": len(PRODUCTS),
        "employees": len(EMPLOYEES),
        "reviews": len(all_review_ratings),
    },
    "aggregations": report_rows,
}
REPORT_PATH = HERE / "aggregation_report.json"
REPORT_PATH.write_text(json.dumps(REPORT, indent=2), encoding="utf-8")

kinds = {row["kind"] for row in report_rows}
types = {row["type"] for row in report_rows}
print()
print("aggregation instances:", len(report_rows))
print("distinct aggregation types:", len(types), "->", ", ".join(sorted(types)))
print("kinds:", sorted(kinds))
print("written:", REPORT_PATH.name, REPORT_PATH.stat().st_size, "bytes")

assert len(report_rows) >= 15
assert kinds == {"bucket", "metric"}
assert len(types) >= 15
assert REPORT_PATH.exists()

  # aggregation                       index     kind    type            value
----------------------------------------------------------------------------------------------------------------------
  1 categories                        products  bucket  terms           12 buckets / 72 docs / top Automotive (6)
  2 top_categories                    products  bucket  terms           10 buckets / 60 docs / top Automotive (6)
  3 average_price                     products  metric  avg             5,112.5067
  4 minimum_price                     products  metric  min             9.5000
  5 maximum_price                     products  metric  max             78,999.0000
  6 price_stats                       products  metric  stats           avg 5,112.51, min 9.50, max 78,999.00
  7 price_percentiles                 products  metric  percentiles     25.0 786.50, 50.0 1,649.00, 75.0 4,124.00, 90.0 11,699.00, 95.0 21,699.00
  8 total_inventory                   products  metric  sum             5

## Bucket vs metric aggregations

### Bucket aggregations

A bucket aggregation **groups documents into buckets**. Each bucket has a `doc_count` and can
contain sub-aggregations. It answers "which groups exist and how many documents are in each?"

| Aggregation | Buckets |
|---|---|
| `terms` | one per distinct value |
| `range` | one per numeric range |
| `date_histogram` | one per time interval |
| `histogram` | one per numeric interval |
| `filters` | several named filter buckets |
| `filter` | **one** bucket with matching documents |
| `missing` | one bucket for documents without the field |
| `nested` | one bucket over nested documents |
| `reverse_nested` | one bucket back at the parent level |
| `global` | one bucket over the whole index, ignoring the query |

Bucket aggregations can contain other bucket aggregations and metric aggregations, which is how
`category -> stats(price)` works. A bucket aggregation never returns a single numeric `value`;
it returns `buckets` and optionally `sum_other_doc_count`.

### Metric aggregations

A metric aggregation **computes numbers from documents** in its scope. It answers "what is the
average/minimum/maximum/count/sum?" and has no buckets.

| Aggregation | Result |
|---|---|
| `avg`, `sum`, `min`, `max` | one number |
| `stats` | count, min, max, avg, sum |
| `extended_stats` | stats + variance/std deviation |
| `value_count` | number of values |
| `cardinality` | approximate distinct count |
| `percentiles` | requested percentile values |

Metric aggregations can be top-level (one number for the whole query) or nested inside a bucket
(one number per bucket). **A metric cannot contain a sub-aggregation**, because it produces a
value, not a group; only bucket aggregations can be parents.

### One-line rule

- **Bucket** = "split into groups" â†’ returns `buckets`.
- **Metric** = "calculate a number" â†’ returns `value`.

The report above mixes both: `terms`/`date_histogram`/`filter` rows are buckets, while
`avg`/`sum`/`stats`/`percentiles`/`cardinality` rows are metrics. Nested metrics such as
`avg_salary_engineering` exist only because the department bucket created a scope for them.

## Verification summary

- 72 products and 120 employees indexed with explicit mappings; `reviews` is `nested`.
- Tasks 1â€“10 each compare Elasticsearch output with a Python calculation and assert equality
  (or the exact expected bucket structure).
- The analytics report contains more than 15 aggregation instances and at least 15 distinct
  aggregation types across both indices.
- `aggregation_report.json` is written next to the notebook.

Run the notebook from this directory with the `es` kernel, or:

```powershell
.venv\Scripts\python.exe -m nbconvert --to notebook --execute --inplace `
  --ExecutePreprocessor.kernel_name=es --ExecutePreprocessor.timeout=180 practical_8.ipynb
```

In [15]:
print("Practical 8 verification")
print("------------------------")
print("products:", product_total, "| employees:", employee_total)
print("product categories:", len(product_result["categories"]["buckets"]))
print("monthly buckets:", len(product_result["monthly_created"]["buckets"]))
print("nested reviews:", product_result["reviews"]["doc_count"])
print("report rows:", len(REPORT["aggregations"]))
print("report file:", REPORT_PATH.name)

assert product_total == 72 and employee_total == 120
assert len(product_result["categories"]["buckets"]) == 12
assert len(product_result["monthly_created"]["buckets"]) == 36
assert product_result["reviews"]["doc_count"] == 181
assert len(REPORT["aggregations"]) >= 15
print("\nAll Practical 8 checks passed.")

Practical 8 verification
------------------------
products: 72 | employees: 120
product categories: 12
monthly buckets: 36
nested reviews: 181
report rows: 42
report file: aggregation_report.json

All Practical 8 checks passed.
